# Zipcode, City, and State Extraction

Goal: 
1. From the mturk_data.csv extract City and State columns reliably with U.S. Zipcode.
2. From the image_meta_data.csv extract Zipcode from lattitude and longitude

## Read csv file

In [3]:
import numpy as np
import pandas as pd
import pgeocode

In [4]:
df = pd.read_csv('../data/mturk_data.csv') # read the CSV file
df.columns # print the column names

Index(['participant_id', 'participant_longitude', 'participant_latitude',
       'participant_zip_code', 'participant_city', 'participant_state', 'age',
       'gender', 'hispanic_or_latino', 'race', 'race_self_identify_text',
       'education_level', 'languages_home', 'languages_home_other',
       'last_flood_experience', 'flood_preparation_actions',
       'flood_response_actions', 'flood_map_experience',
       'flood_map_experience_other', 'selected_folder', 'selected_image',
       'image_longitude', 'image_latitude', 'image_zip_code', 'image_city',
       'image_state', 'flood_risk_severity', 'flood_frequency',
       'flood_risk_understanding', 'confidence_in_map',
       'likely_to_buy_property', 'likely_to_rent_property',
       'likely_to_buy_insurance', 'likely_to_take_preventive_actions',
       'use_flood_maps_frequency', 'ease_of_interpretation',
       'additional_comments'],
      dtype='object')

## 1.Extract City and State from participant_zipcode column 

In [5]:
# make sure ZIP is a clean 5-digit string
zip_clean = (
    df["participant_zip_code"]
      .astype(str)
      .str.strip()
      .str.extract(r"(\d{5})")[0]
      .str.zfill(5)
)

In [6]:
nomi = pgeocode.Nominatim("US")

def get_city_state(zip_code):
    if pd.isna(zip_code):
        return pd.Series({"participant_city": np.nan,
                          "participant_state": np.nan})
    rec = nomi.query_postal_code(zip_code)
    # rec is a Series; access fields safely
    place = rec.get("place_name")
    state = rec.get("state_name")
    if pd.isna(place) and pd.isna(state):
        # pgeocode didn’t find it
        return pd.Series({"participant_city": np.nan,
                          "participant_state": np.nan})
    city = str(place).split(",")[0].strip() if place else np.nan
    return pd.Series({"participant_city": city,
                      "participant_state": state})

# apply to cleaned ZIPs
city_state = zip_clean.apply(get_city_state)

df["participant_city"] = city_state["participant_city"]
df["participant_state"] = city_state["participant_state"]

The above code block extracts the city and state name for the `participant_zipcode` 

# 2. Extract Zipcode form longitude and latitude of the Image

In [7]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [8]:
image_df = pd.read_csv('../data/image_meta_data.csv') # load data
image_df.columns

Index(['Image File Name', 'Country', 'State', 'City', 'Zipcode', 'Longitude',
       'Latitude', 'Flood Name', 'Date Taken ', 'Web Link (Image)',
       'Last Accessed', 'Source', 'Web Link (Google Street View)',
       'Direct Link', 'Working Web Link (Image) ', 'Comment', 'Unnamed: 16'],
      dtype='object')

The following code demonstrates how Zipcodes are extracted from geographic coordinates using the **`geopy`** library and OpenStreetMap’s **Nominatim** API.

This method performs **reverse geocoding** to convert latitude and longitude pairs into human-readable address components such as Zipcodes.  
A rate limiter is applied to ensure compliance with OpenStreetMap’s usage policy (maximum of one request per second).

In [9]:
# Initialize geolocator 
geolocator = Nominatim(user_agent="zipcode_lookup")

# Add a rate limiter (1 req/sec)
geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_zipcode(row):
    lat, lon = row["Latitude"], row["Longitude"]
    if pd.isna(lat) or pd.isna(lon):
        return None
    try:
        location = geocode((lat, lon), exactly_one=True, addressdetails=True)
        if location and "address" in location.raw:
            return location.raw["address"].get("postcode")
    except Exception:
        return None
    return None

# Apply the function to each row 
image_df["ZIPcode"] = image_df.apply(get_zipcode, axis=1)

# Clean and zero-pad
image_df["ZIPcode"] = image_df["ZIPcode"].astype(str).str.extract(r"(\d{5})")[0].str.zfill(5)

# Save
# image_df.to_csv("with_zipcodes.csv", index=False) # Uncomment to save the results
print("Zipcodes added:", image_df["ZIPcode"].notna().sum(), "/", len(image_df))

Zipcodes added: 257 / 261
